# Tratamiento de datos en el CSV

In [17]:
import pandas as pd
import numpy as np

# Cargar el archivo generado por Mockaroo
df = pd.read_csv("sample_data/citas_2025_metropolitan.csv")

# La carga del csv genera un ; al final del nombre de la columna totalpagado
if 'totalpagado;' in df.columns:
    df.rename(columns={'totalpagado;': 'totalpagado'}, inplace=True)

# También es generado para todos sus registros, por lo que se elimina de ambos lados
if 'totalpagado' in df.columns:
    df['totalpagado'] = df['totalpagado'].astype(str).str.replace(';', '', regex=False)
    df['totalpagado'] = pd.to_numeric(df['totalpagado'], errors='coerce')

print(f"Registros iniciales cargados: {len(df)}")

Registros iniciales cargados: 10000


### Normalización y repeticiones de pacientes, doctores y ID

In [18]:
# Crear artificialmente repeticiones de nombres de doctores y pacientes
num_doctores_fijos = 200 # Num de doctores escogidos para que se repartan en el dataset
doctores_unicos = df['doctor'].unique()[:num_doctores_fijos]
df['doctor'] = np.random.choice(doctores_unicos, size=len(df))

num_pacientes_unicos = 800 # Num de paceintes escogidos para que se repartan en el dataset
pacientes_unicos = df['paciente'].unique()[:num_pacientes_unicos]
df['paciente'] = np.random.choice(pacientes_unicos, size=len(df))

# En Mockaroo hacemos 10 veces el csv para tener 10000 registros, esto hace que los ID se repitan
df['ID_cit'] = range(1, len(df) + 1) # Iniciará un conteo desde 1 hasta el máximo de filas para reasignar IDs

# Convertir columna de fecha al tipo datetime correcto para ordenar
df['fechacit'] = pd.to_datetime(df['fechacit'])

# Ordenar cronológicamente por fecha de cita
df = df.sort_values(by='fechacit').reset_index(drop=True)

print(f"Cantidad de registros: {len(df)}")
print(df)

Cantidad de registros: 10000
      ID_cit   fechacit      estado             paciente            doctor  \
0       6445 2025-01-01  Completada        Korie Gratten  Marisa McManamen   
1       1093 2025-01-01  Completada       Gonzales Tatem     Auroora Hardy   
2       4472 2025-01-01   Cancelada   Ursuline Berntssen  Philippe Jolliff   
3       5921 2025-01-01  Completada      Winn Berrisford    Katalin Tootin   
4       8296 2025-01-01  Completada       Base Shrimpton   Gherardo Badger   
...      ...        ...         ...                  ...               ...   
9995    9818        NaT   Cancelada  Kerianne Gatteridge      Jandy Doding   
9996    9825        NaT  Completada         Anthe Fright    Yorgos Larvent   
9997    9870        NaT  Completada       Nathan Everwin       Issie Toler   
9998    9978        NaT   Cancelada       Camile Goullee    Laney Crosetto   
9999   10000        NaT  Completada         Brett Onians      Mina Camplen   

                sucursal  pagopaci

### Completado de datos faltantes

In [19]:
# Caso A: Eliminar registros donde 'fecha_cit' sea nulo ya que no son utilizables
df = df.dropna(subset=['fechacit']).copy()

# Caso B: Colocar 0.00 a los nulos en 'PagoAseguradora' (pacientes sin seguro)
df['pagoaseguradora'] = df['pagoaseguradora'].fillna(0.00)

# Caso C: Imputar los nulos de 'estado' asumiendo que la cita se completó (Moda)
df['estado'] = df['estado'].fillna('Completada')

print(f"Cantidad de registros: {len(df)}")
print(df)

Cantidad de registros: 9407
      ID_cit   fechacit      estado            paciente             doctor  \
0       6445 2025-01-01  Completada       Korie Gratten   Marisa McManamen   
1       1093 2025-01-01  Completada      Gonzales Tatem      Auroora Hardy   
2       4472 2025-01-01   Cancelada  Ursuline Berntssen   Philippe Jolliff   
3       5921 2025-01-01  Completada     Winn Berrisford     Katalin Tootin   
4       8296 2025-01-01  Completada      Base Shrimpton    Gherardo Badger   
...      ...        ...         ...                 ...                ...   
9402    1476 2025-12-30  Completada        Daren Stodit    Sidonnie Helder   
9403    3957 2025-12-30  Completada        Buck Hankard    Yehudi Duinkerk   
9404    3961 2025-12-30  Completada       Orazio Ivetts    Koralle Quinlan   
9405    2769 2025-12-30  Completada        Pat Broschek  Ellette Bertholin   
9406    7122 2025-12-30   Cancelada      Devlen Camillo       Jandy Doding   

                sucursal  pagopacie

### Inconsistencias

In [20]:
# Regla A: Citas canceladas no deben registrar pagos (Forzar a 0.00)
df.loc[df['estado'] == 'Cancelada', ['pagopaciente', 'pagoaseguradora']] = 0.00

# Regla B: Recalcular estrictamente el TotalPagado con base en la corrección
df['totalpagado'] = df['pagopaciente'] + df['pagoaseguradora']

columnas_pago = ['pagopaciente', 'pagoaseguradora', 'totalpagado']
df[columnas_pago] = df[columnas_pago].round(2)

print(f"Cantidad de registros: {len(df)}")
print(df)

Cantidad de registros: 9407
      ID_cit   fechacit      estado            paciente             doctor  \
0       6445 2025-01-01  Completada       Korie Gratten   Marisa McManamen   
1       1093 2025-01-01  Completada      Gonzales Tatem      Auroora Hardy   
2       4472 2025-01-01   Cancelada  Ursuline Berntssen   Philippe Jolliff   
3       5921 2025-01-01  Completada     Winn Berrisford     Katalin Tootin   
4       8296 2025-01-01  Completada      Base Shrimpton    Gherardo Badger   
...      ...        ...         ...                 ...                ...   
9402    1476 2025-12-30  Completada        Daren Stodit    Sidonnie Helder   
9403    3957 2025-12-30  Completada        Buck Hankard    Yehudi Duinkerk   
9404    3961 2025-12-30  Completada       Orazio Ivetts    Koralle Quinlan   
9405    2769 2025-12-30  Completada        Pat Broschek  Ellette Bertholin   
9406    7122 2025-12-30   Cancelada      Devlen Camillo       Jandy Doding   

                sucursal  pagopacie

### Exportación del dataset

In [21]:
# Guardamos el archivo final en una carpeta de datos para el proyecto
df.to_csv("citas_2025_metropolitan_limpio.csv", index=False)

print(f"Registros finales después de la limpieza: {len(df)}")
print("¡El CSV limpio ha sido generado con éxito para el Dashboard!")

Registros finales después de la limpieza: 9407
¡El CSV limpio ha sido generado con éxito para el Dashboard!
